# VideoForge — GPU Generation Server (Google Colab, free T4)

Runs an **open-weight text→video / image→video model (LTX-Video)** on Colab's
free GPU and exposes it over a public URL. VideoForge's `remote` provider calls
it, so `generate_video(..., provider='remote')` produces real footage that
composites into your timeline.

**Steps:** Runtime → Change runtime type → **GPU (T4)**, then Run all. Copy the
printed `VIDEOFORGE_VIDEOGEN_URL` and set it where VideoForge runs:
```bash
export VIDEOFORGE_VIDEOGEN_URL='https://xxxx.trycloudflare.com'
export VIDEOFORGE_VIDEOGEN_PROVIDER=remote
```
Keep this tab running — closing it stops the server.

## 1. Check GPU

In [ ]:
!nvidia-smi -L || echo 'No GPU! Set Runtime -> Change runtime type -> GPU'

## 2. Install dependencies

In [ ]:
%pip -q install "diffusers>=0.32" "transformers>=4.44" accelerate safetensors \
    imageio imageio-ffmpeg sentencepiece fastapi "uvicorn[standard]" nest-asyncio pillow
print('deps installed')

## 3. Load LTX-Video (t2v + i2v)
Fits a free T4 via CPU offload. First run downloads weights (a few minutes).

In [ ]:
import torch, gc
from diffusers import LTXPipeline, LTXImageToVideoPipeline

MODEL_ID = 'Lightricks/LTX-Video'
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

t2v = LTXPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
t2v.enable_model_cpu_offload()
# share the loaded components with the image-to-video pipeline (no 2nd download)
i2v = LTXImageToVideoPipeline(**t2v.components)
i2v.enable_model_cpu_offload()
print('LTX-Video ready:', MODEL_ID)

## 4. Generation API
Implements the contract VideoForge's `RemoteHTTPProvider` speaks:
`GET /health` and `POST /generate` → raw `video/mp4`.

In [ ]:
import base64, io, tempfile, math
from PIL import Image
from fastapi import FastAPI, Request, Response
from diffusers.utils import export_to_video

def _snap(v, m, lo):
    return max(lo, int(round(v / m)) * m)

def run_generation(body: dict) -> bytes:
    prompt = body.get('prompt', '')
    mode = body.get('mode', 't2v')
    fps = float(body.get('fps', 24))
    # LTX needs spatial dims divisible by 32 and frames == 8*k + 1
    w = _snap(int(body.get('width', 768)), 32, 32)
    h = _snap(int(body.get('height', 432)), 32, 32)
    nf = int(body.get('num_frames', 97))
    nf = max(9, int(round((nf - 1) / 8)) * 8 + 1)
    steps = int(body.get('steps', 30))
    guidance = float(body.get('guidance', 3.0))
    seed = body.get('seed')
    gen = torch.Generator(device='cuda').manual_seed(int(seed)) if seed is not None else None
    kw = dict(prompt=prompt, width=w, height=h, num_frames=nf,
              num_inference_steps=steps, guidance_scale=guidance, generator=gen)
    if mode == 'i2v' and body.get('image_b64'):
        img = Image.open(io.BytesIO(base64.b64decode(body['image_b64']))).convert('RGB')
        frames = i2v(image=img.resize((w, h)), **kw).frames[0]
    else:
        frames = t2v(**kw).frames[0]
    path = tempfile.mktemp(suffix='.mp4')
    export_to_video(frames, path, fps=fps)
    with open(path, 'rb') as f:
        return f.read()

app = FastAPI()

@app.get('/health')
def health():
    return {'ok': True, 'model': MODEL_ID, 'modes': ['t2v', 'i2v'],
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

@app.post('/generate')
async def generate(request: Request):
    body = await request.json()
    data = run_generation(body)
    return Response(content=data, media_type='video/mp4')

print('API defined')

## 5. Public tunnel + launch
Starts the server and a free Cloudflare quick-tunnel (no account needed).

In [ ]:
import nest_asyncio, threading, uvicorn, subprocess, re, time, os, urllib.request, stat
nest_asyncio.apply()

def _serve():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)

# download cloudflared and open a quick tunnel
BIN = '/usr/local/bin/cloudflared'
if not os.path.exists(BIN):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', BIN)
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC)

proc = subprocess.Popen([BIN, 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    m = re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break

print('\n' + '=' * 60)
print('VIDEOFORGE_VIDEOGEN_URL =', url)
print('=' * 60)
print('On the machine running VideoForge:')
print(f"  export VIDEOFORGE_VIDEOGEN_URL='{url}'")
print('  export VIDEOFORGE_VIDEOGEN_PROVIDER=remote')
print('Keep this tab open to keep the server alive.')

## 6. (Optional) Smoke test the endpoint

In [ ]:
import requests, json
r = requests.get(url + '/health', timeout=30); print('health:', r.json())
r = requests.post(url + '/generate', json={'prompt': 'a golden retriever running on a beach, slow motion',
    'mode': 't2v', 'num_frames': 49, 'width': 704, 'height': 480, 'fps': 24, 'steps': 30}, timeout=900)
open('test.mp4', 'wb').write(r.content); print('wrote test.mp4', len(r.content), 'bytes')
from IPython.display import Video; Video('test.mp4', embed=True)